# Validação de Dados com Pydantic e IntFlag

Este notebook demonstra como utilizar a biblioteca **Pydantic** para validação de modelos de dados e o uso de **IntFlag** para gestão de permissões/papéis de utilizador de forma eficiente.

In [1]:
from enum import auto, IntFlag
from typing import Any
from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    SecretStr,
    ValidationError,
)

## 1. Definição de Papéis (Roles)

Utilizamos `IntFlag` para permitir a combinação de múltiplos papéis (bitmask). O `auto()` atribui potências de 2 automaticamente.

In [2]:
class Role(IntFlag):
    # Author = 1, Editor = 2, Developer = 4
    Author = auto()
    Editor = auto()
    Developer = auto()
    
    # Admin é uma combinação de Author, Editor e Developer (1 | 2 | 4 = 7)
    Admin = Author | Editor | Developer

## 2. Modelo de Utilizador (Pydantic)

O Pydantic valida tipos e formatos automaticamente. Aqui definimos:
- `EmailStr`: Valida o formato do email.
- `SecretStr`: Protege dados sensíveis (não aparece em logs).
- `Field`: Adiciona metadados e restrições (ex: `frozen=True`).

In [3]:
class User(BaseModel):
    name: str = Field(examples=["Arjan"])
    email: EmailStr = Field(
        examples=["example@arjancodes.com"],
        description="The email address of the user",
        frozen=True,
    )
    password: SecretStr = Field(
        examples=["Password123"], description="The password of the user"
    )
    role: Role = Field(default=None, description="The role of the user")

## 3. Função de Validação

Esta função tenta validar um dicionário contra o modelo `User` e captura erros de validação.

In [4]:
def validate(data: dict[str, Any]) -> None:
    try:
        user = User.model_validate(data)
        print("Usuário válido:")
        print(user)
    except ValidationError as e:
        print("Usuário inválido!")
        for error in e.errors():
            print(f"- {error['loc']}: {error['msg']}")

## 4. Demonstração Prática

Vamos testar com dados corretos e incorretos.

In [5]:
# Dados válidos
good_data = {
    "name": "Arjan",
    "email": "example@arjancodes.com",
    "password": "Password123",
}

print("--- Testando Dados Válidos ---")
validate(good_data)

print("\n--- Testando Dados Inválidos ---")
# Dados inválidos (email incorreto e falta de campos)
bad_data = {"email": "<bad data>", "password": "<bad data>"}
validate(bad_data)

--- Testando Dados Válidos ---
Usuário válido:
name='Arjan' email='example@arjancodes.com' password=SecretStr('**********') role=None

--- Testando Dados Inválidos ---
Usuário inválido!
- ('name',): Field required
- ('email',): value is not a valid email address: An email address must have an @-sign.
